In [ ]:
# Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def parse_conll_file(conll_file_path):
    data = []
    tokens = []
    ner_tags = []

    # Unified tag mapping (B- and I- merged)
    tag2id = {
        "O": 0,
        "Claim": 1,
        "Premise": 2
    }

    with open(conll_file_path, 'r') as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if line == "":
                if tokens:
                    data.append({
                        'id': str(len(data)),
                        'ner_tags': ner_tags,
                        'tokens': tokens
                    })
                    tokens = []
                    ner_tags = []
            else:
                parts = line.split()
                if len(parts) != 3:
                    print(f"Skipping malformed line {i}: {line}")
                    continue

                token, _, tag = parts
                tokens.append(token)

                # Merge B- and I- tags
                if tag in ["B-Claim", "I-Claim"]:
                    ner_tags.append(tag2id["Claim"])
                elif tag in ["B-Premise", "I-Premise"]:
                    ner_tags.append(tag2id["Premise"])
                elif tag == "O":
                    ner_tags.append(tag2id["O"])
                else:
                    print(f"Unknown tag '{tag}' found, skipping line: {line}")
                    continue

        if tokens:
            data.append({
                'id': str(len(data)),
                'ner_tags': ner_tags,
                'tokens': tokens
            })

    return data

In [ ]:
conll_file_path = "/path/to/your/file"
micro_docs = parse_conll_file(conll_file_path)

In [ ]:
# Validation: check if only expected labels are used
all_labels = {label for entry in micro_docs for label in entry["ner_tags"]}
unexpected_labels = all_labels - {0, 1, 2, 3, 4}
if unexpected_labels:
    print(f"Warning: Found unexpected label IDs: {unexpected_labels}")

In [ ]:
for entry in micro_docs:
    print(entry)

In [ ]:
!pip install transformers
!pip install accelerate -U
!pip install datasets

In [ ]:
!pip install bitsandbytes accelerate

In [ ]:
# Mapping for conversion
id2label = {0: "O", 1: "Claim", 2: "Premise"}
label2id = {"O": 0, "Claim": 1, "Premise": 2}

In [ ]:
# A function to attach punctuation to sentences
import re

def reconstruct_sentence(tokens):
    sentence = ""
    for i, token in enumerate(tokens):
        if re.match(r"^[.,!?;:’”')\]]$", token):
            sentence += token
        elif token in ['(', '[', '“', '‘']:
            if sentence and not sentence.endswith(" "):
                sentence += " "
            sentence += token
        else:
            if sentence and not sentence.endswith(" "):
                sentence += " "
            sentence += token
    return sentence

In [ ]:
# Creating a df with sentences instead of separate tokens
import pandas as pd

records = []
for sent in micro_docs:
    sentence_txt = reconstruct_sentence(sent["tokens"])
    label_txt = " ".join(id2label[t] for t in sent["ner_tags"])
    records.append({"sentence": sentence_txt, "ner_tag": label_txt})

df = pd.DataFrame(records)

In [ ]:
print(df.head(5))

                                            sentence  \
0  -DOCSTART- Yes, it 's annoying and cumbersome ...   
1  One can hardly move in Friedrichshain or Neukö...   
2  Health insurance companies should not cover tr...   
3  Of course there are a number of programmes in ...   
4  Intelligence services must urgently be regulat...   

                                             ner_tag  
0  O Claim Claim Claim Claim Claim Claim Claim Cl...  
1  Premise Premise Premise Premise Premise Premis...  
2  Claim Claim Claim Claim Claim Claim Claim Clai...  
3  Premise Premise Premise Premise Premise Premis...  
4  Claim Claim Claim Claim Claim Claim Claim Clai...  


In [ ]:
from huggingface_hub import login

# Make sure you are logged in to Hugging Face
# You can authenticate by running: huggingface-cli login
# Or by passing your token: login(token="YOUR_TOKEN")
login()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

In [ ]:
def build_prompt(sentence):
    prompt = f"""Your task is to analyze the following text and label each sentence as *Claim* or *Premise*.
Claim is a concluding statement.
Premise is a piece of evidence, a fact, that supports or attacks a claim.
The text around each sentence is the context.

Instructions:
- For each Claim or Premise sentence found, return a separate JSON object with exactly two fields:
  - "sentence": the exact span from the input sentence expressing the argument.
  - "type": "Claim" or "Premise".
- If none of them is found, return one JSON object with both fields set to "".
- Do **not** wrap the JSON objects in a list (no square brackets).
- Separate multiple JSON objects with **commas and spaces only**, e.g.:
  {{ "sentence": "...", "type": "Claim" }}, {{ "sentence": "...", "type": "Premise" }}
- The output must be strictly valid JSON:
  - Use double quotes only
  - Close all braces correctly
  - Do not include trailing commas
- Do not include any explanation, notes, or extra text. Output **only** the JSON objects.

Sentence:
{sentence}
"""
    return prompt

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", quantization_config=bnb_config)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512, do_sample=False)

In [ ]:
predictions = []

for row in tqdm(df.itertuples(), total=len(df)):
    prompt = build_prompt(row.sentence)
    output = pipe(prompt)[0]['generated_text']

    output_clean = output.replace(prompt, "").strip()

    predictions.append({
        "sentence": row.sentence,
        "gold": row.ner_tag,
        "mistral_output": output_clean
    })

In [ ]:
import json

In [ ]:
output_path = "/path/to/your/project/microtext_premise_claim_predictions.json"
with open(output_path, "w") as f:
    json.dump(predictions, f, indent=2)

print(f"Saved predictions to {output_path}")

In [ ]:
with open("/path/to/your/project/microtext_premise_claim_predictions.json") as f:
    predictions = json.load(f)

In [ ]:
print(predictions[1]["sentence"])
print(predictions[1]["gold"])
print(predictions[1]["mistral_output"])

In [ ]:
import json
import nltk
from difflib import SequenceMatcher
from sklearn.metrics import classification_report
from nltk.tokenize import word_tokenize, sent_tokenize

nltk.download('punkt_tab')

__________________________
__________________________
__________________________

In [ ]:
import re

def extract_mistral_sentences(mistral_output):
    mistral_output = mistral_output.strip()

    # Remove the "Output:" prefix if present
    if mistral_output.lower().startswith("output:"):
        mistral_output = mistral_output[len("output:"):].strip()

    # Extract JSON-like pairs: {"sentence": "...", "type": "..."}
    pattern = r'\{\s*"sentence"\s*:\s*"(.+?)"\s*,\s*"type"\s*:\s*"(.+?)"\s*\}'
    matches = re.findall(pattern, mistral_output, re.DOTALL)

    return [(s.strip(), t.strip().lower()) for s, t in matches]

def get_sentence_level_labels(text, gold_labels):
    tokens = word_tokenize(text.strip())  # precise tokenization
    labels = gold_labels.strip().split()
    assert len(tokens) == len(labels), f"Token and label length mismatch: {len(tokens)} vs {len(labels)}"

    sentences = sent_tokenize(text)
    sentence_labels = []

    start = 0
    for sent in sentences:
        sent_tokens = word_tokenize(sent)
        sent_labels = labels[start:start+len(sent_tokens)]
        start += len(sent_tokens)

        if "Claim" in sent_labels:
            label = "Claim"
        elif "Premise" in sent_labels:
            label = "Premise"
        else:
            label = "O"
        sentence_labels.append((sent.strip(), label))

    return sentence_labels

def best_match(pred_sent, gold_sents):
    pred_clean = pred_sent.lower()
    best_idx, best_score = -1, 0
    for i, (gold_sent, _) in enumerate(gold_sents):
        score = SequenceMatcher(None, pred_clean, gold_sent.lower()).ratio()
        if score > best_score:
            best_idx, best_score = i, score
    return best_idx if best_score > 0.8 else -1  # threshold

gold_all = []
pred_all = []

for item in predictions:
    gold_pairs = get_sentence_level_labels(item["sentence"], item["gold"])
    pred_pairs = extract_mistral_sentences(item["mistral_output"])

    matched = set()
    pred_sent_map = {}

    for pred_sent, pred_label in pred_pairs:
        idx = best_match(pred_sent, gold_pairs)
        if idx != -1 and idx not in matched:
            pred_sent_map[idx] = pred_label
            matched.add(idx)

    for i, (gold_sent, gold_label) in enumerate(gold_pairs):
      pred_label = pred_sent_map.get(i, "O")  # assign "O" if not predicted

      # Normalize both labels to have capitalized first letter
      gold_label = gold_label.strip().capitalize()
      pred_label = pred_label.strip().capitalize()

      gold_all.append(gold_label)
      pred_all.append(pred_label)

print(classification_report(
    gold_all,
    pred_all,
    labels=["Claim", "Premise", "O"],
    digits=4
))

In [ ]:
# Choose any entry to inspect
entry = predictions[5]

gold_sentences = get_sentence_level_labels(entry["sentence"], entry["gold"])
print("\nGold sentence-level labels:")
for s, l in gold_sentences:
    print(f"- [{l.upper()}] {s}")

pred_spans = extract_mistral_sentences(entry["mistral_output"])
print("\nPredicted sentences from Mistral:")
for s, l in pred_spans:
    print(f"- [{l.upper()}] {s}")

In [ ]:
# from tqdm import tqdm

# # Load predictions
# with open("/path/to/your/project/microtext_premise_claim_predictions.json") as f:
#     predictions = json.load(f)

# # Inspect mismatches
# mismatch_indices = []

# for i, entry in enumerate(tqdm(predictions)):
#     tokens = word_tokenize(entry["sentence"].strip())
#     labels = entry["gold"].strip().split()

#     if len(tokens) != len(labels):
#         print(f"\n❌ Mismatch at index {i}")
#         print(f"Sentence: {entry['sentence']}")
#         print(f"Tokens ({len(tokens)}): {tokens}")
#         print(f"Labels ({len(labels)}): {labels}")
#         print(f"Difference: {len(tokens) - len(labels)} tokens")
#         mismatch_indices.append(i)

# print(f"\n🔍 Total mismatches found: {len(mismatch_indices)} out of {len(predictions)}")